## Chat Models - <a href='https://python.langchain.com/docs/modules/data_connection/document_loaders/'>Document Loaders</a> and Text Splitting


In [ ]:
%pip install -q langchain langchain-openai langchain-core langchain-community beautifulsoup4 requests

In [ ]:
import os
from getpass import getpass

# API Key Setup - uses environment variable or prompts for input
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

In [ ]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import TextLoader
import requests

# Get this file and save it locally:
url = "https://github.com/hammer-mt/thumb/blob/master/README.md"

# Save it locally:
r = requests.get(url)

# Extract the text from the HTML:
soup = BeautifulSoup(r.text, 'html.parser')
text = soup.get_text()

with open("README.md", "w") as f:
    f.write(text)

loader = TextLoader('README.md')
docs = loader.load()

In [ ]:
from langchain_core.documents import Document
[ Document(page_content='test', metadata={'test': 'test'}) ] 

In [ ]:
# Split the text into sentences:
from langchain.text_splitter import RecursiveCharacterTextSplitter

CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size = CHUNK_SIZE,
    chunk_overlap  = CHUNK_OVERLAP,
    length_function = len,
    is_separator_regex = False,
)

final_docs = text_splitter.split_documents(loader.load())

In [ ]:
len(final_docs)

In [ ]:
from langchain_openai.chat_models import ChatOpenAI
from langchain.chains.summarize import load_summarize_chain

chat = ChatOpenAI()

In [ ]:
chain = load_summarize_chain(llm=chat, chain_type="map_reduce")
chain.invoke({
    "input_documents": final_docs,
})